In [ ]:
# Configure
from pathlib import Path
from runpy import run_path

TIER = "M1"  # Both notebook tiers use the BTC CRM roster.
BLUEPRINT_COUNT = None  # M2 defaults to 150. Set a higher count if needed.
M1_BLUEPRINTS = None  # Optional path to the M1 blueprint JSONL for M2.
INPUT_ROOT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/output")


In [ ]:
# Load the attached Stage 1 input dataset
generators = list(INPUT_ROOT.rglob("stage-1/src/generate.py"))
if len(generators) != 1:
    raise ValueError("Attach one Stage 1 dataset with the input ZIP extracted.")
generator = run_path(str(generators[0]))
customers = generator["load_crm_seed_registry"]() if TIER == "M1" else None


In [ ]:
# Generate, validate, and save the blueprints and customer registry
options = {}
if TIER == "M2":
    if "M2_CRM_PLAN" not in generator:
        raise ValueError("Attach the updated stage1-m2-input.zip dataset to run M2.")
    options = dict(blueprint_count=BLUEPRINT_COUNT,
                   m1_blueprints=Path(M1_BLUEPRINTS) if M1_BLUEPRINTS else None)
registry, blueprints, sessions = generator["build_dataset"](TIER, crm_customers=customers, **options)
OUTPUT.mkdir(parents=True, exist_ok=True)
generator["write_dataset"](OUTPUT, registry, blueprints)
print(f"{TIER}: {len(registry)} customers, {len(blueprints)} journeys, {sessions} sessions in {OUTPUT}")
